# Phase 04S: Teacher C → S3 Logit-KD Three-Seed Aggregate

Tổng hợp chính thức ba seed định trước `42 / 123 / 2026`:

1. khóa ba full run và xác minh toàn bộ manifest/checksum;
2. kiểm tra sáu checkpoint và protocol consistency;
3. so sánh paired baseline S3 CE-only với direct Logit-KD;
4. tính mean, population SD, sample SD và paired delta;
5. tổng hợp learning curves và ba biểu đồ;
6. áp dụng reproduction gate chỉ trên Validation;
7. giữ Held-out Test `SEALED`;
8. nếu đạt, khép lại KD reproduction và authorize Phase 5.

## Kaggle inputs

1. Phase 04Q Teacher C → S3 seed 42 completed output.
2. Phase 04R Teacher C → S3 seed 123 completed output.
3. Phase 04R Teacher C → S3 seed 2026 completed output.

CPU là đủ. Internet Off. Không cần gắn MS COCO, Phase 1, Phase 2 hoặc
checkpoint model nguồn khác. Notebook không deserialize checkpoint.


## Cell 1 — Discover và khóa ba formal runs

Phát hiện đúng authorization marker và đúng một complete marker cho mỗi seed;
xác minh experiment ID, trạng thái hoàn tất, progress boundary và Test boundary.

**Đầu ra dự kiến:** `3 / 3 DISCOVERED`, Test Set `SEALED`.


In [1]:
import csv
import hashlib
import json
import math
import os
import statistics
import time
from datetime import datetime, timezone
from pathlib import Path


CELL_STARTED_AT = time.perf_counter()
INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path(
    "/kaggle/working//"
    "phase4_teacher_c_s3_logit_kd_three_seed_aggregate"
)
AUDIT_DIR = OUTPUT_ROOT / "audits"
RESULT_DIR = OUTPUT_ROOT / "results"
FIGURE_DIR = OUTPUT_ROOT / "figures"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
for directory in [
    OUTPUT_ROOT, AUDIT_DIR, RESULT_DIR, FIGURE_DIR, MANIFEST_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

SEEDS = (42, 123, 2026)
EXPERIMENT_ID = "P4C_C_S3_LOGIT"


def now_utc():
    return datetime.now(timezone.utc).isoformat()


def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def exactly_one(name):
    matches = sorted(INPUT_ROOT.rglob(name))
    assert len(matches) == 1, (
        f"Cần đúng một {name}; tìm thấy {len(matches)}: {matches}"
    )
    return matches[0].resolve()


def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    target = (root / relative_path).resolve()
    assert target.is_relative_to(root), (root, target)
    return target


CONFIRMATION_AUTHORIZED_PATH = exactly_one(
    "PHASE4_TEACHER_C_S3_LOGIT_KD_"
    "CONFIRMATION_SEEDS_AUTHORIZED.json"
)
confirmation_authorized = read_json(
    CONFIRMATION_AUTHORIZED_PATH
)
assert confirmation_authorized["status"] == "authorized"
assert confirmation_authorized["experiment_id"] == EXPERIMENT_ID
assert int(
    confirmation_authorized["development_seed_complete"]
) == 42
assert confirmation_authorized["confirmation_seeds"] == [123, 2026]
assert confirmation_authorized["method_changes_allowed"] is False
assert confirmation_authorized["test_set_status"] == "sealed"
assert confirmation_authorized["test_set_used"] is False

RUNS = {}
for seed in SEEDS:
    marker_path = exactly_one(
        "PHASE4_TEACHER_C_S3_LOGIT_KD_"
        f"SEED{seed}_FULL_RUN_COMPLETE.json"
    )
    marker = read_json(marker_path)
    assert marker["status"] == "complete"
    assert marker["experiment_id"] == EXPERIMENT_ID
    assert int(marker["seed"]) == seed
    assert int(marker["completed_epochs"]) >= 5
    assert int(marker["progress"]["epoch"]) == int(
        marker["completed_epochs"]
    )
    assert int(marker["progress"]["cursor"]) == 0
    assert marker["manifest_verification"] == "all_passed"
    assert marker["test_set_status"] == "sealed"
    assert marker["test_set_used"] is False
    if seed == 42:
        assert marker["confirmation_seeds_authorized"] is True
    else:
        assert marker[
            "ready_for_phase04s_three_seed_aggregate"
        ] is True
    RUNS[seed] = {
        "root": marker_path.parent,
        "marker_path": marker_path,
        "marker": marker,
    }

write_json(
    AUDIT_DIR / "phase4s_cell01_input_authorization.json",
    {
        "audit_version": "phase4s_cell01_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "experiment_id": EXPERIMENT_ID,
        "seeds": list(SEEDS),
        "formal_runs": "3 / 3",
        "confirmation_authorization": "verified",
        "method_changes_allowed": False,
        "dataset_read": False,
        "caption_read": False,
        "checkpoint_payload_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04S CELL 1 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Formal runs               : 3 / 3 DISCOVERED")
print("Seeds                     : 42 / 123 / 2026")
print("Confirmation authorization: VERIFIED")
print("Method changes allowed    : False")
print("Dataset/caption read      : False / False")
print("Checkpoint payload read   : False")
print("Test Set status           : SEALED")
print("Progress                  : 1 / 6 cells passed")
print("Next                      : manifests and protocol")


PHASE 04S CELL 1 STATUS
Status                    : PASSED
Formal runs               : 3 / 3 DISCOVERED
Seeds                     : 42 / 123 / 2026
Confirmation authorization: VERIFIED
Method changes allowed    : False
Dataset/caption read      : False / False
Checkpoint payload read   : False
Test Set status           : SEALED
Progress                  : 1 / 6 cells passed
Next                      : manifests and protocol


## Cell 2 — Toàn vẹn manifest, checkpoint và protocol

Đọc manifest của từng run, tính lại size/SHA-256 cho mọi artifact, xác minh
SHA256SUMS và sáu checkpoint, sau đó so sánh toàn bộ trường protocol khóa.

**Đầu ra dự kiến:** `157 / 157`, checkpoint `6 / 6`, protocol `EXACT`.


In [2]:
CELL_STARTED_AT = time.perf_counter()
MANIFEST_COUNTS = {}
CONTRACTS = {}
contract_fields = [
    "optimizer", "learning_rates", "weight_decay",
    "physical_batch", "accumulation", "effective_batch",
    "steps_per_epoch", "maximum_epochs", "total_steps",
    "warmup_steps", "minimum_lr_factor", "mixed_precision",
    "gradscaler_initial_scale", "overflow_retry_limit",
    "backward_divisor", "gradient_clip", "teacher_frozen",
    "temperature", "alpha", "test_set_status", "test_set_used",
]

for seed in SEEDS:
    run = RUNS[seed]
    root = run["root"]
    marker = run["marker"]
    manifest_path = resolve_inside(
        root, marker["manifest"]["csv"]["relative_path"]
    )
    assert sha256_file(manifest_path) == (
        marker["manifest"]["csv"]["sha256"]
    )
    rows = list(csv.DictReader(
        manifest_path.open("r", encoding="utf-8", newline="")
    ))
    assert len(rows) == int(marker["manifest_rows"])
    expected_sum_lines = []
    for row in rows:
        artifact = resolve_inside(root, row["relative_path"])
        assert artifact.stat().st_size == int(row["size_bytes"])
        assert sha256_file(artifact) == row["sha256"]
        expected_sum_lines.append(
            f'{row["sha256"]}  {row["relative_path"]}'
        )
    sums_path = resolve_inside(
        root,
        marker["manifest"]["sha256sums"]["relative_path"],
    )
    assert sha256_file(sums_path) == (
        marker["manifest"]["sha256sums"]["sha256"]
    )
    actual_sum_lines = [
        line.strip()
        for line in sums_path.read_text(
            encoding="utf-8"
        ).splitlines()
        if line.strip()
    ]
    assert actual_sum_lines == expected_sum_lines

    for role in ("best", "last"):
        checkpoint = resolve_inside(
            root, marker[f"{role}_checkpoint"]["relative_path"]
        )
        assert sha256_file(checkpoint) == (
            marker[f"{role}_checkpoint"]["sha256"]
        )

    contract_matches = sorted(
        (root / "configs").glob(
            "phase4*_production_optimizer_contract.json"
        )
    )
    assert len(contract_matches) == 1, contract_matches
    contract = read_json(contract_matches[0])
    assert contract["status"] == "locked"
    assert contract["test_set_status"] == "sealed"
    assert contract["test_set_used"] is False
    CONTRACTS[seed] = {
        field: contract[field] for field in contract_fields
    }
    MANIFEST_COUNTS[seed] = len(rows)

reference_contract = CONTRACTS[42]
for seed in (123, 2026):
    assert CONTRACTS[seed] == reference_contract

assert MANIFEST_COUNTS == {42: 53, 123: 52, 2026: 52}
assert sum(MANIFEST_COUNTS.values()) == 157
write_json(
    AUDIT_DIR / "phase4s_cell02_integrity_contract.json",
    {
        "audit_version": "phase4s_cell02_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "manifest_rows": MANIFEST_COUNTS,
        "total_manifest_rows": 157,
        "checkpoint_file_hashes": "6 / 6 verified",
        "protocol_fields_compared": contract_fields,
        "protocol_consistency": "exact",
        "teacher_frozen": reference_contract["teacher_frozen"],
        "temperature": reference_contract["temperature"],
        "alpha": reference_contract["alpha"],
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04S CELL 2 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Manifest rows             :", MANIFEST_COUNTS)
print("Total rows verified       : 157 / 157")
print("Checkpoint hashes         : 6 / 6 VERIFIED")
print("Protocol consistency      : EXACT")
print("Teacher frozen            :", reference_contract["teacher_frozen"])
print(
    "Temperature / alpha       :",
    reference_contract["temperature"], "/",
    reference_contract["alpha"],
)
print("Test Set status           : SEALED")
print("Progress                  : 2 / 6 cells passed")
print("Next                      : paired three-seed analysis")


PHASE 04S CELL 2 STATUS
Status                    : PASSED
Manifest rows             : {42: 53, 123: 52, 2026: 52}
Total rows verified       : 157 / 157
Checkpoint hashes         : 6 / 6 VERIFIED
Protocol consistency      : EXACT
Teacher frozen            : True
Temperature / alpha       : 4.0 / 0.5
Test Set status           : SEALED
Progress                  : 2 / 6 cells passed
Next                      : paired three-seed analysis


## Cell 3 — Phân tích paired ba seed

Ghép đúng seed giữa baseline CE-only và KD, tính CIDEr-D, BLEU-4, Validation CE,
paired delta, mean, độ lệch chuẩn và kiểm tra mọi đại lượng hữu hạn.

**Gate:** mỗi seed phải có `ΔCIDEr-D ≥ 0.001`, BLEU-4 tăng và CE giảm.


In [3]:
CELL_STARTED_AT = time.perf_counter()


def summary(values):
    values = [float(value) for value in values]
    return {
        "n": len(values),
        "mean": statistics.mean(values),
        "population_std": statistics.pstdev(values),
        "sample_std": statistics.stdev(values),
        "minimum": min(values),
        "maximum": max(values),
    }


SEED_ROWS = []
for seed in SEEDS:
    marker = RUNS[seed]["marker"]
    comparison = marker["comparison"]
    baseline = comparison[f"baseline_seed{seed}"]
    kd = comparison[f"kd_seed{seed}"]
    delta = comparison["delta"]
    best = marker["best_result"]
    assert int(kd["best_epoch"]) == int(best["epoch"])
    assert math.isclose(
        float(kd["CIDEr-D"]),
        float(best["metrics"]["CIDEr-D"]),
        rel_tol=0.0,
        abs_tol=1e-12,
    )
    assert comparison["selection_split"] == "validation"
    assert comparison["test_set_status"] == "sealed"
    assert comparison["test_set_used"] is False
    row = {
        "seed": seed,
        "completed_epochs": int(marker["completed_epochs"]),
        "best_epoch": int(best["epoch"]),
        "overflow_retries_at_best": int(
            best["overflow_retries"]
        ),
        "baseline_cider_d": float(baseline["CIDEr-D"]),
        "kd_cider_d": float(kd["CIDEr-D"]),
        "delta_cider_d": float(delta["CIDEr-D"]),
        "relative_cider_gain_percent": (
            100.0 * float(delta["CIDEr-D"])
            / float(baseline["CIDEr-D"])
        ),
        "baseline_bleu_4": float(baseline["BLEU-4"]),
        "kd_bleu_4": float(kd["BLEU-4"]),
        "delta_bleu_4": float(delta["BLEU-4"]),
        "baseline_validation_ce": float(
            baseline["validation_ce"]
        ),
        "kd_validation_ce": float(kd["validation_ce"]),
        "delta_validation_ce": float(
            delta["validation_ce"]
        ),
        "kd_bleu_1": float(best["metrics"]["BLEU-1"]),
        "kd_meteor": float(best["metrics"]["METEOR"]),
        "kd_rouge_l": float(best["metrics"]["ROUGE-L"]),
        "all_primary_directions_improved": (
            float(delta["CIDEr-D"]) > 0.0
            and float(delta["BLEU-4"]) > 0.0
            and float(delta["validation_ce"]) < 0.0
        ),
        "cider_gate_plus_0_001": (
            float(delta["CIDEr-D"]) >= 0.001
        ),
    }
    assert all(
        math.isfinite(value)
        for key, value in row.items()
        if isinstance(value, float)
    )
    SEED_ROWS.append(row)

PER_SEED_CSV = RESULT_DIR / (
    "teacher_c_s3_logit_kd_three_seed_results.csv"
)
with PER_SEED_CSV.open(
    "w", encoding="utf-8", newline=""
) as handle:
    writer = csv.DictWriter(
        handle, fieldnames=list(SEED_ROWS[0])
    )
    writer.writeheader()
    writer.writerows(SEED_ROWS)

metric_pairs = {
    "CIDEr-D": ("baseline_cider_d", "kd_cider_d", "delta_cider_d"),
    "BLEU-4": ("baseline_bleu_4", "kd_bleu_4", "delta_bleu_4"),
    "validation_ce": (
        "baseline_validation_ce",
        "kd_validation_ce",
        "delta_validation_ce",
    ),
}
AGGREGATE = {}
for metric, (baseline_key, kd_key, delta_key) in metric_pairs.items():
    AGGREGATE[metric] = {
        "baseline": summary([
            row[baseline_key] for row in SEED_ROWS
        ]),
        "kd": summary([row[kd_key] for row in SEED_ROWS]),
        "paired_delta": summary([
            row[delta_key] for row in SEED_ROWS
        ]),
    }

AGGREGATE["KD_only_caption_metrics"] = {
    "BLEU-1": summary([row["kd_bleu_1"] for row in SEED_ROWS]),
    "METEOR": summary([row["kd_meteor"] for row in SEED_ROWS]),
    "ROUGE-L": summary([row["kd_rouge_l"] for row in SEED_ROWS]),
}
AGGREGATE["relative_cider_gain_percent"] = summary([
    row["relative_cider_gain_percent"] for row in SEED_ROWS
])

cider_wins = sum(
    row["delta_cider_d"] > 0.0 for row in SEED_ROWS
)
all_primary_wins = sum(
    row["all_primary_directions_improved"]
    for row in SEED_ROWS
)
# P(X >= 3 | n=3, p=0.5) = 0.125. Two-sided exact = 0.25.
sign_test = {
    "n": 3,
    "positive_cider_deltas": cider_wins,
    "one_sided_exact_p": 0.125 if cider_wins == 3 else None,
    "two_sided_exact_p": 0.25 if cider_wins == 3 else None,
    "interpretation": (
        "descriptive only; n=3 is insufficient for a "
        "strong inferential claim"
    ),
}

SUMMARY_PATH = RESULT_DIR / (
    "teacher_c_s3_logit_kd_three_seed_summary.json"
)
write_json(
    SUMMARY_PATH,
    {
        "schema_version": "phase4s_three_seed_summary_v1",
        "created_at_utc": now_utc(),
        "experiment_id": EXPERIMENT_ID,
        "seeds": list(SEEDS),
        "per_seed": SEED_ROWS,
        "aggregate": AGGREGATE,
        "paired_consistency": {
            "cider_wins": f"{cider_wins} / 3",
            "all_primary_direction_wins": (
                f"{all_primary_wins} / 3"
            ),
            "sign_test": sign_test,
        },
        "selection_split": "validation",
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

write_json(
    AUDIT_DIR / "phase4s_cell03_metrics.json",
    {
        "audit_version": "phase4s_cell03_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "finite_metrics": True,
        "paired_seed_alignment": "3 / 3",
        "cider_improvement_gate": sum(
            row["cider_gate_plus_0_001"]
            for row in SEED_ROWS
        ),
        "all_primary_direction_wins": all_primary_wins,
        "sign_test_interpretation": sign_test["interpretation"],
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04S PAIRED THREE-SEED RESULTS")
print("=" * 72)
for row in SEED_ROWS:
    print(
        f"seed={row['seed']:4d} | "
        f"CIDEr-D {row['baseline_cider_d']:.6f} -> "
        f"{row['kd_cider_d']:.6f} "
        f"({row['delta_cider_d']:+.6f}) | "
        f"BLEU-4 {row['delta_bleu_4']:+.6f} | "
        f"CE {row['delta_validation_ce']:+.6f}"
    )
print("-" * 72)
print(
    "Mean CIDEr-D baseline/KD :",
    f"{AGGREGATE['CIDEr-D']['baseline']['mean']:.6f} / "
    f"{AGGREGATE['CIDEr-D']['kd']['mean']:.6f}",
)
print(
    "Mean paired CIDEr delta  :",
    f"{AGGREGATE['CIDEr-D']['paired_delta']['mean']:+.6f}",
)
print(
    "Mean relative gain       :",
    f"{AGGREGATE['relative_cider_gain_percent']['mean']:+.2f}%",
)
print(
    "CIDEr-D population SD    :",
    f"{AGGREGATE['CIDEr-D']['baseline']['population_std']:.6f} -> "
    f"{AGGREGATE['CIDEr-D']['kd']['population_std']:.6f}",
)
print("All primary improvements  :", f"{all_primary_wins} / 3")
print("Test Set status           : SEALED")
print("Next                      : aggregate curves and figures")


PHASE 04S PAIRED THREE-SEED RESULTS
seed=  42 | CIDEr-D 1.076799 -> 1.097713 (+0.020915) | BLEU-4 +0.006322 | CE -0.034283
seed= 123 | CIDEr-D 1.076710 -> 1.095957 (+0.019247) | BLEU-4 +0.012333 | CE -0.029352
seed=2026 | CIDEr-D 1.070902 -> 1.091982 (+0.021080) | BLEU-4 +0.009884 | CE -0.043232
------------------------------------------------------------------------
Mean CIDEr-D baseline/KD : 1.074804 / 1.095217
Mean paired CIDEr delta  : +0.020414
Mean relative gain       : +1.90%
CIDEr-D population SD    : 0.002759 -> 0.002398
All primary improvements  : 3 / 3
Test Set status           : SEALED
Next                      : aggregate curves and figures


## Cell 4 — Learning curves và biểu đồ

Tổng hợp 20 epoch của ba seed, xuất CSV đường học cùng ba biểu đồ:
CIDEr-D theo epoch, Validation CE theo epoch và baseline-vs-KD CIDEr-D.


In [4]:
CELL_STARTED_AT = time.perf_counter()
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


HISTORY_BY_SEED = {}
for seed in SEEDS:
    history_matches = sorted(
        (RUNS[seed]["root"] / "logs").glob(
            f"teacher_c_s3_logit_kd_seed{seed}_epoch_history.csv"
        )
    )
    assert len(history_matches) == 1
    rows = list(csv.DictReader(
        history_matches[0].open(
            "r", encoding="utf-8", newline=""
        )
    ))
    assert len(rows) == int(
        RUNS[seed]["marker"]["completed_epochs"]
    )
    assert [int(row["epoch"]) for row in rows] == list(
        range(1, len(rows) + 1)
    )
    HISTORY_BY_SEED[seed] = rows

maximum_common_epoch = min(
    len(rows) for rows in HISTORY_BY_SEED.values()
)
curve_metrics = [
    "train_combined_loss", "train_ce", "train_kd_kl_t2",
    "validation_ce", "BLEU-4", "CIDEr-D",
]
CURVE_ROWS = []
for epoch in range(1, maximum_common_epoch + 1):
    row = {"epoch": epoch}
    for metric in curve_metrics:
        values = [
            float(HISTORY_BY_SEED[seed][epoch - 1][metric])
            for seed in SEEDS
        ]
        row[f"{metric}_mean"] = statistics.mean(values)
        row[f"{metric}_population_std"] = (
            statistics.pstdev(values)
        )
    CURVE_ROWS.append(row)

CURVE_CSV = RESULT_DIR / (
    "teacher_c_s3_logit_kd_three_seed_epoch_curves.csv"
)
with CURVE_CSV.open(
    "w", encoding="utf-8", newline=""
) as handle:
    writer = csv.DictWriter(
        handle, fieldnames=list(CURVE_ROWS[0])
    )
    writer.writeheader()
    writer.writerows(CURVE_ROWS)

epochs = list(range(1, maximum_common_epoch + 1))
colors = {42: "#2563eb", 123: "#16a34a", 2026: "#dc2626"}

fig, ax = plt.subplots(figsize=(10, 6))
for seed in SEEDS:
    ax.plot(
        epochs,
        [
            float(row["CIDEr-D"])
            for row in HISTORY_BY_SEED[seed][
                :maximum_common_epoch
            ]
        ],
        label=f"Seed {seed}",
        color=colors[seed],
        linewidth=1.8,
    )
ax.set_title("Teacher C → S3 Logit-KD: Validation CIDEr-D")
ax.set_xlabel("Epoch")
ax.set_ylabel("CIDEr-D")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
CIDER_FIGURE = FIGURE_DIR / "phase4s_cider_by_epoch.png"
fig.savefig(CIDER_FIGURE, dpi=180)
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 6))
for seed in SEEDS:
    ax.plot(
        epochs,
        [
            float(row["validation_ce"])
            for row in HISTORY_BY_SEED[seed][
                :maximum_common_epoch
            ]
        ],
        label=f"Seed {seed}",
        color=colors[seed],
        linewidth=1.8,
    )
ax.set_title("Teacher C → S3 Logit-KD: Validation CE")
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-Entropy (dimensionless; lower is better)")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
CE_FIGURE = FIGURE_DIR / "phase4s_validation_ce_by_epoch.png"
fig.savefig(CE_FIGURE, dpi=180)
plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 6))
x_positions = list(range(len(SEEDS)))
width = 0.34
ax.bar(
    [x - width / 2 for x in x_positions],
    [row["baseline_cider_d"] for row in SEED_ROWS],
    width,
    label="CE-only baseline",
    color="#94a3b8",
)
ax.bar(
    [x + width / 2 for x in x_positions],
    [row["kd_cider_d"] for row in SEED_ROWS],
    width,
    label="Logit-KD",
    color="#2563eb",
)
ax.set_xticks(x_positions, [str(seed) for seed in SEEDS])
ax.set_xlabel("Seed")
ax.set_ylabel("Validation CIDEr-D")
ax.set_title("Paired baseline vs. Logit-KD")
ax.grid(axis="y", alpha=0.25)
ax.legend()
fig.tight_layout()
PAIRED_FIGURE = FIGURE_DIR / (
    "phase4s_baseline_vs_kd_cider.png"
)
fig.savefig(PAIRED_FIGURE, dpi=180)
plt.close(fig)

for figure in [CIDER_FIGURE, CE_FIGURE, PAIRED_FIGURE]:
    assert figure.is_file()
    assert figure.stat().st_size > 10_000

write_json(
    AUDIT_DIR / "phase4s_cell04_curves_figures.json",
    {
        "audit_version": "phase4s_cell04_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "seeds": list(SEEDS),
        "common_epochs": maximum_common_epoch,
        "curve_rows": len(CURVE_ROWS),
        "figures": {
            path.name: {
                "size_bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
            for path in [
                CIDER_FIGURE, CE_FIGURE, PAIRED_FIGURE
            ]
        },
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04S CELL 4 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Seed histories            : 3 / 3")
print("Common epochs             :", maximum_common_epoch)
print("Aggregate curve rows      :", len(CURVE_ROWS))
print("Figures                   : 3 / 3 CREATED")
print("Test Set status           : SEALED")
print("Next                      : reproduction gate and package")


PHASE 04S CELL 4 STATUS
Status                    : PASSED
Seed histories            : 3 / 3
Common epochs             : 20
Aggregate curve rows      : 20
Figures                   : 3 / 3 CREATED
Test Set status           : SEALED
Next                      : reproduction gate and package


## Cell 5 — Reproduction gate và Phase 5 authorization

Khóa kết luận trên Validation. Chỉ authorize Phase 5 nếu cả ba seed cùng đạt
CIDEr gate, ba hướng chỉ số chính, protocol exact và finite metrics.


In [5]:
CELL_STARTED_AT = time.perf_counter()
cider_gate_count = sum(
    row["cider_gate_plus_0_001"] for row in SEED_ROWS
)
all_direction_count = sum(
    row["all_primary_directions_improved"]
    for row in SEED_ROWS
)
protocol_exact = all(
    CONTRACTS[seed] == CONTRACTS[42]
    for seed in (123, 2026)
)
finite_gate = all(
    math.isfinite(value)
    for row in SEED_ROWS
    for value in row.values()
    if isinstance(value, float)
)
REPRODUCTION_PASSED = (
    cider_gate_count == 3
    and all_direction_count == 3
    and protocol_exact
    and finite_gate
)
assert REPRODUCTION_PASSED

CONCLUSION_PATH = RESULT_DIR / (
    "phase4s_reproduction_conclusion.json"
)
write_json(
    CONCLUSION_PATH,
    {
        "schema_version": "phase4s_reproduction_conclusion_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "experiment_id": EXPERIMENT_ID,
        "teacher": "Teacher C — SigLIP ViT-B/16",
        "student": "S3 — TinyViT-5M",
        "method": "direct response/logit KD",
        "temperature": 4.0,
        "alpha": 0.5,
        "seeds": list(SEEDS),
        "gates": {
            "cider_delta_at_least_0_001": (
                f"{cider_gate_count} / 3"
            ),
            "all_primary_directions_improved": (
                f"{all_direction_count} / 3"
            ),
            "protocol_consistency": "exact",
            "finite_metrics": True,
            "test_set_status": "sealed",
            "test_set_used": False,
        },
        "mean_results": {
            "baseline_cider_d": AGGREGATE[
                "CIDEr-D"
            ]["baseline"]["mean"],
            "kd_cider_d": AGGREGATE[
                "CIDEr-D"
            ]["kd"]["mean"],
            "paired_cider_delta": AGGREGATE[
                "CIDEr-D"
            ]["paired_delta"]["mean"],
            "relative_cider_gain_percent": AGGREGATE[
                "relative_cider_gain_percent"
            ]["mean"],
            "baseline_bleu_4": AGGREGATE[
                "BLEU-4"
            ]["baseline"]["mean"],
            "kd_bleu_4": AGGREGATE[
                "BLEU-4"
            ]["kd"]["mean"],
            "paired_bleu_4_delta": AGGREGATE[
                "BLEU-4"
            ]["paired_delta"]["mean"],
            "baseline_validation_ce": AGGREGATE[
                "validation_ce"
            ]["baseline"]["mean"],
            "kd_validation_ce": AGGREGATE[
                "validation_ce"
            ]["kd"]["mean"],
            "paired_validation_ce_delta": AGGREGATE[
                "validation_ce"
            ]["paired_delta"]["mean"],
        },
        "stability": {
            "baseline_cider_population_std": AGGREGATE[
                "CIDEr-D"
            ]["baseline"]["population_std"],
            "kd_cider_population_std": AGGREGATE[
                "CIDEr-D"
            ]["kd"]["population_std"],
            "kd_is_more_stable_on_cider": (
                AGGREGATE["CIDEr-D"]["kd"]["population_std"]
                < AGGREGATE["CIDEr-D"][
                    "baseline"
                ]["population_std"]
            ),
        },
        "statistical_caution": (
            "Only three prespecified seeds are available. "
            "The sign test is descriptive and does not establish "
            "a strong inferential claim."
        ),
        "selection_split": "validation",
        "reproduction_conclusion": (
            "Teacher C -> S3 direct Logit-KD reproduction passed "
            "on all three prespecified Validation seeds."
        ),
        "next_phase": (
            "Phase 05A KD improvement entry and cross-pair comparison"
        ),
    },
)

NEXT_AUTHORIZATION = OUTPUT_ROOT / (
    "PHASE5_KD_IMPROVEMENT_ENTRY_AUTHORIZED.json"
)
write_json(
    NEXT_AUTHORIZATION,
    {
        "marker_version": (
            "phase5_kd_improvement_entry_authorized_v1"
        ),
        "created_at_utc": now_utc(),
        "status": "authorized",
        "source_experiment": EXPERIMENT_ID,
        "source_reproduction_status": "passed",
        "completed_reproduction_pairs": [
            "Teacher A -> S1 response/logit KD",
            "Teacher B -> S2 sequence KD",
            "Teacher C -> S3 direct response/logit KD",
        ],
        "required_first_step": (
            "Phase 05A lock the improvement study, paired baselines, "
            "Validation gates and ablation budget"
        ),
        "method_changes_allowed": False,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

write_json(
    AUDIT_DIR / "phase4s_cell05_reproduction_gate.json",
    {
        "audit_version": "phase4s_cell05_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "reproduction_passed": REPRODUCTION_PASSED,
        "cider_gate_count": cider_gate_count,
        "all_primary_direction_count": all_direction_count,
        "protocol_exact": protocol_exact,
        "finite_metrics": finite_gate,
        "phase5_authorized": True,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04S REPRODUCTION GATE")
print("=" * 72)
print("Status                    : PASSED")
print("CIDEr-D >= +0.001         :", f"{cider_gate_count} / 3")
print("CIDEr/BLEU/CE directions  :", f"{all_direction_count} / 3")
print("Protocol consistency      : EXACT")
print("Finite metrics            : True")
print("Mean CIDEr-D delta        :", (
    AGGREGATE["CIDEr-D"]["paired_delta"]["mean"]
))
print("KD CIDEr-D more stable    :", (
    AGGREGATE["CIDEr-D"]["kd"]["population_std"]
    < AGGREGATE["CIDEr-D"]["baseline"]["population_std"]
))
print("Test Set status           : SEALED")
print("Phase 05A KD improvement  : AUTHORIZED")
print("Progress                  : 5 / 6 cells passed")
print("Next                      : formal package")


PHASE 04S REPRODUCTION GATE
Status                    : PASSED
CIDEr-D >= +0.001         : 3 / 3
CIDEr/BLEU/CE directions  : 3 / 3
Protocol consistency      : EXACT
Finite metrics            : True
Mean CIDEr-D delta        : 0.02041371843742934
KD CIDEr-D more stable    : True
Test Set status           : SEALED
Phase 05A KD improvement  : AUTHORIZED
Progress                  : 5 / 6 cells passed
Next                      : formal package


## Cell 6 — Formal package

Tạo manifest CSV/JSON/SHA256SUMS, complete marker và liên kết tới summary,
conclusion cùng authorization Phase 5.


In [6]:
CELL_STARTED_AT = time.perf_counter()
MANIFEST_CSV = MANIFEST_DIR / "phase4s_manifest.csv"
MANIFEST_JSON = MANIFEST_DIR / "phase4s_manifest.json"
SHA256SUMS = MANIFEST_DIR / "phase4s_SHA256SUMS.txt"
FINAL_MARKER = OUTPUT_ROOT / (
    "PHASE4_TEACHER_C_S3_LOGIT_KD_"
    "THREE_SEED_AGGREGATE_COMPLETE.json"
)

artifact_paths = sorted(
    [
        path for path in OUTPUT_ROOT.rglob("*")
        if path.is_file()
        and path not in {
            MANIFEST_CSV, MANIFEST_JSON, SHA256SUMS, FINAL_MARKER
        }
        and not path.name.endswith(".tmp")
    ],
    key=lambda path: path.relative_to(OUTPUT_ROOT).as_posix(),
)
manifest_rows = [
    {
        "relative_path": path.relative_to(
            OUTPUT_ROOT
        ).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in artifact_paths
]
with MANIFEST_CSV.open(
    "w", encoding="utf-8", newline=""
) as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["relative_path", "size_bytes", "sha256"],
    )
    writer.writeheader()
    writer.writerows(manifest_rows)
write_json(
    MANIFEST_JSON,
    {
        "schema_version": "phase4s_manifest_v1",
        "created_at_utc": now_utc(),
        "artifact_count": len(manifest_rows),
        "artifacts": manifest_rows,
    },
)
SHA256SUMS.write_text(
    "".join(
        f'{row["sha256"]}  {row["relative_path"]}\n'
        for row in manifest_rows
    ),
    encoding="utf-8",
)
for row in manifest_rows:
    artifact = resolve_inside(
        OUTPUT_ROOT, row["relative_path"]
    )
    assert artifact.stat().st_size == int(row["size_bytes"])
    assert sha256_file(artifact) == row["sha256"]

write_json(
    FINAL_MARKER,
    {
        "marker_version": (
            "phase4_teacher_c_s3_logit_kd_"
            "three_seed_aggregate_complete_v1"
        ),
        "created_at_utc": now_utc(),
        "status": "complete",
        "experiment_id": EXPERIMENT_ID,
        "teacher": "Teacher C — SigLIP ViT-B/16",
        "student": "S3 — TinyViT-5M",
        "method": "direct response/logit KD",
        "seeds": list(SEEDS),
        "source_manifest_rows_verified": 157,
        "protocol_consistency": "exact",
        "reproduction_status": "passed",
        "cider_gate": "3 / 3",
        "all_primary_direction_gate": "3 / 3",
        "summary": {
            "relative_path": SUMMARY_PATH.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(SUMMARY_PATH),
        },
        "conclusion": {
            "relative_path": CONCLUSION_PATH.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(CONCLUSION_PATH),
        },
        "next_authorization": {
            "relative_path": NEXT_AUTHORIZATION.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(NEXT_AUTHORIZATION),
        },
        "manifest": {
            "csv": {
                "relative_path": MANIFEST_CSV.relative_to(
                    OUTPUT_ROOT
                ).as_posix(),
                "sha256": sha256_file(MANIFEST_CSV),
            },
            "json": {
                "relative_path": MANIFEST_JSON.relative_to(
                    OUTPUT_ROOT
                ).as_posix(),
                "sha256": sha256_file(MANIFEST_JSON),
            },
            "sha256sums": {
                "relative_path": SHA256SUMS.relative_to(
                    OUTPUT_ROOT
                ).as_posix(),
                "sha256": sha256_file(SHA256SUMS),
            },
        },
        "manifest_rows": len(manifest_rows),
        "manifest_verification": "all_passed",
        "selection_split": "validation",
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

print("=" * 72)
print("PHASE 04S CELL 6 STATUS")
print("=" * 72)
print("Status                    : COMPLETE")
print("Source runs               : 3 / 3 VERIFIED")
print("Source manifest rows      : 157 / 157")
print("Reproduction gate         : PASSED")
print("Aggregate artifact rows   :", len(manifest_rows))
print("Manifest verification     : ALL PASSED")
print("Figures                   : 3")
print("Selection split           : Validation")
print("Test Set status           : SEALED")
print("Phase 05A KD improvement  : AUTHORIZED")
print("Progress                  : 6 / 6 cells passed")
print("Marker                    :", FINAL_MARKER)


PHASE 04S CELL 6 STATUS
Status                    : COMPLETE
Source runs               : 3 / 3 VERIFIED
Source manifest rows      : 157 / 157
Reproduction gate         : PASSED
Aggregate artifact rows   : 13
Manifest verification     : ALL PASSED
Figures                   : 3
Selection split           : Validation
Test Set status           : SEALED
Phase 05A KD improvement  : AUTHORIZED
Progress                  : 6 / 6 cells passed
Marker                    : /kaggle/working//phase4_teacher_c_s3_logit_kd_three_seed_aggregate/PHASE4_TEACHER_C_S3_LOGIT_KD_THREE_SEED_AGGREGATE_COMPLETE.json
